In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

# Load API key from .env file into environment
load_dotenv()

# Initialize the Anthropic client (reads ANTHROPIC_API_KEY from env automatically)
client = Anthropic()
# Use Haiku for fast, cost-effective tool use responses
model = "claude-haiku-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message


# Handles both raw strings and API Message objects when appending to history
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        # If message is a Message object, use its .content; otherwise treat as raw string/list
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


# Sends the conversation to Claude; tools param enables tool use when provided
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    # Only add optional params if they have values
    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    # Returns the full Message object (not just text) so we can inspect tool_use blocks
    message = client.messages.create(**params)
    return message


# Extracts only text blocks from a response, ignoring tool_use blocks
def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# Tools and Schemas
# Each tool needs: (1) a Python function and (2) a JSON schema describing it for Claude

from datetime import datetime, timedelta


# Adds a duration to a datetime string, handling edge cases like leap years and month lengths
def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    # Parse the input string into a datetime object using the specified format
    date = datetime.strptime(datetime_str, input_format)

    # Use timedelta for simple time units
    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        # Manual month arithmetic since timedelta doesn't support months
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        # Clamp day to max valid day for the target month (handles Feb 30 -> Feb 28, etc.)
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    # Return in a human-readable format with day-of-week, full month name, and AM/PM
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


# Simulates setting a reminder — in production this would persist to a database
def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


# JSON Schema: tells Claude what add_duration_to_datetime expects and returns
add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

# JSON Schema: tells Claude what set_reminder expects
set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

# Batch tool: a meta-tool that lets Claude invoke multiple tools in a single call
# Avoids sequential round-trips when tools are independent of each other
batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [ ]:
# Step 1 - Write a tool function 

from datetime import datetime
from anthropic.types import ToolParam

# Step 1 - Write a tool function that returns the current date/time
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    # Format the current system time using Python's strftime codes
    return datetime.now().strftime(date_format)

# Step 2 - Write a JSON schema so Claude knows how to call this tool
# ToolParam is a typed wrapper from the SDK that adds validation
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

# Step 3 - Call Claude with JSON schema attached as available tools

messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

# Claude sees the tool schema and decides whether to call it
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

# Add Claude's response (which contains a tool_use block) to conversation history
# Note: response.content can have multiple blocks (text + tool_use)
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

In [ ]:
# Step 4 - Run Tool locally using the arguments Claude provided
# Unpack the tool input dict as keyword arguments to the actual Python function
result = get_current_datetime(**response.content[0].input)

result

In [ ]:
# Step 5 - Add Tool Result block and call Claude again
# The tool result must be sent as a "user" message with type "tool_result"

messages.append({
    "role": "user",
    "content": [
        {
        "type":"tool_result",
        "tool_use_id": response.content[0].id,  # Must match the tool_use block's ID
        "content": result,                        # The actual output from running the tool
        "is_error": False                         # Set True if the tool execution failed
        }
    ]
})

print(messages)

# Final call: Claude now has the tool result and can formulate a natural language answer
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [ ]:
# Test call: using the helper functions instead of manual message construction

messages = []

add_user_message(messages, "What is the current time in HH:MM:SS format?")

# Claude will respond with a tool_use block requesting get_current_datetime
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

# add_assistant_message handles Message objects by extracting .content automatically
add_assistant_message(messages, response.content)

print(messages)

The complete multi-turn conversation works like this:

Send user message to Claude with available tools<br>
Claude responds with text and/or tool requests<br>
Execute all requested tools and create result blocks<br>
Send tool results back as a user message<br>
Repeat until Claude provides a final answer

In [ ]:
import json


# Dispatches a tool call by name to the corresponding Python function
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)


# Processes all tool_use blocks from Claude's response, executes them, and builds tool_result blocks
def run_tools(message):
    # Filter for only tool_use blocks (response may also contain text blocks)
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            # Execute the tool with the arguments Claude provided
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,  # Links result back to the specific tool call
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            # Return errors gracefully so Claude can recover or inform the user
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [ ]:
# The Agentic Conversation Loop
# Keeps calling Claude until it stops requesting tools (i.e., has a final answer)

def run_conversation(messages):
    """Loop: send message -> Claude responds -> if tool_use, execute tools & feed results back -> repeat"""
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)

        # Print any text Claude included in the response
        print(text_from_message(response))

        # stop_reason == "tool_use" means Claude wants to call a tool before giving a final answer
        # Any other stop_reason (e.g. "end_turn") means the conversation is complete
        if response.stop_reason != "tool_use":
            break

        # Execute all requested tools and send results back as a user message
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [ ]:
# Test: Claude should make two parallel tool calls (one for HH:MM, one for SS)
messages = []
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?",
)
run_conversation(messages)